In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)

In [5]:
from sklearn.metrics import accuracy_score, f1_score

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [8]:
dataset = load_dataset("AI-Lab-Makerere/beans", split="train[:600]")
val_dataset = load_dataset("AI-Lab-Makerere/beans", split="validation[:150]")

README.md:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  144MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.5MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.7MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/133 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

In [9]:
labels = dataset.features["labels"].names

In [10]:
labels

['angular_leaf_spot', 'bean_rust', 'healthy']

In [12]:
label2id = {label: i for i, label in enumerate(labels)}

label2id

{'angular_leaf_spot': 0, 'bean_rust': 1, 'healthy': 2}

In [13]:
id2label = {i: label for i, label in enumerate(labels)}

id2label

{0: 'angular_leaf_spot', 1: 'bean_rust', 2: 'healthy'}

In [14]:
model_ckpt = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(model_ckpt)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

In [24]:
print("Training columns:", dataset.column_names)
print("Validation columns:", val_dataset.column_names)

print("\nTraining features:")
print(dataset.features)

print("\nFirst training example:")
print(dataset[0])

Training columns: ['image_file_path', 'image', 'labels']
Validation columns: ['image_file_path', 'image', 'labels']

Training features:
{'image_file_path': Value('string'), 'image': Image(mode=None, decode=True), 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'])}

First training example:
{'image_file_path': '/home/albert/.cache/huggingface/datasets/downloads/extracted/967f0d9f61a7a8de58892c6fab6f02317c06faf3e19fba6a07b0885a9a7142c7/train/angular_leaf_spot/angular_leaf_spot_train.0.jpg', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x500 at 0x7C7C6C48CC20>, 'labels': 0}


In [16]:
def transform(example_batch):
    inputs = image_processor([x for x in example_batch["image"]], return_tensors="pt")
    inputs["labels"] = example_batch["labels"]
    return inputs

In [17]:
encoded_train = dataset.with_transform(transform)
encoded_val = val_dataset.with_transform(transform)

In [19]:
model = AutoModelForImageClassification.from_pretrained(
    model_ckpt,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1": f1}

In [28]:
training_args = TrainingArguments(
    output_dir="./vit_beans_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False
    
)

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    compute_metrics=compute_metrics,
)

In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.523213,1.086515,0.661654,0.538360
2,0.282218,1.070979,0.661654,0.535879
3,0.150371,1.079002,0.661654,0.542802
4,0.106051,1.125138,0.661654,0.535879
5,0.071128,1.170500,0.661654,0.547661
6,0.058208,1.201482,0.661654,0.545791
7,0.049707,1.233106,0.661654,0.545791
8,0.044390,1.257913,0.661654,0.545791
9,0.040905,1.283458,0.661654,0.547661
10,0.037492,1.299105,0.661654,0.547661


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (13 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vit.layers.0.attention.q_proj.weight', 'vit.layers.0.attention.q_proj.bias', 'vit.layers.0.attention.k_proj.weight', 'vit.layers.0.attention.k_proj.bias', 'vit.layers.0.attention.v_proj.weight', 'vit.layers.0.attention.v_proj.bias', 'vit.layers.0.attention.o_proj.weight', 'vit.layers.0.attention.o_proj.bias', 'vit.layers.0.layernorm_before.weight', 'vit.layers.0.layernorm_before.bias', 'vit.layers.0.layernorm_after.weight', 'vit.layers.0.layernorm_after.bias', 'vit.layers.0.mlp.fc1.weight', 'vit.layers.0.mlp.fc1.bias', 'vit.layers.0.mlp.fc2.weight', 'vit.layers.0.mlp.fc2.bias', 'vit.layers.1.attention.q_proj.weight', 'vit.layers.1.attention.q_proj.bias', 'vit.layers.1.attention.k_proj.weight', 'vit.layers.1.attention.k_proj.bias', 'vit.layers.1.attention.v_proj.weight', 'vit.layers.1.attention.v_proj.bias', 'vit.layers.1.attention.o_proj.weight', 'vit.layers.1.attention.o_proj.bias', 'vit.layers.1.layernorm_before

TrainOutput(global_step=380, training_loss=0.07876103810573878, metrics={'train_runtime': 387.2626, 'train_samples_per_second': 30.987, 'train_steps_per_second': 0.981, 'total_flos': 9.29912209809408e+17, 'train_loss': 0.07876103810573878, 'epoch': 20.0})

In [31]:
save_directory = "./saved_vit_beans_model"
trainer.save_model(save_directory)
image_processor.save_pretrained(save_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['./saved_vit_beans_model/preprocessor_config.json']

In [33]:
from transformers import pipeline
vision_classifier = pipeline("image-classification", model=save_directory)

sample_image = val_dataset[0]["image"]
actual_label = labels[val_dataset[0]["labels"]]

prediction = vision_classifier(sample_image)

print(f"Actual Condition: {actual_label}")
print(f"Model Prediction: {prediction}")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Actual Condition: angular_leaf_spot
Model Prediction: [{'label': 'angular_leaf_spot', 'score': 0.9566580057144165}, {'label': 'healthy', 'score': 0.022313041612505913}, {'label': 'bean_rust', 'score': 0.021028930321335793}]


In [39]:
sample_image = val_dataset[42]["image"]
actual_label = labels[val_dataset[42]["labels"]]

prediction = vision_classifier(sample_image)

print(f"Actual Condition: {actual_label}")
print(f"Model Prediction: {prediction}")

Actual Condition: angular_leaf_spot
Model Prediction: [{'label': 'angular_leaf_spot', 'score': 0.9528145790100098}, {'label': 'bean_rust', 'score': 0.025393035262823105}, {'label': 'healthy', 'score': 0.02179240621626377}]
